# Using genomic-benchmarks-data4agents

This notebook demonstrates dataset discovery, metadata, loading, caching, and quality-control (QC) results with the agent-safe package. It exposes train and validation (`val`) data where available, but never test data or test-related QC.

Install the package if it is not already available in this environment:

```python
%pip install genomic-benchmarks-data4agents
```

In [ ]:
from html import escape
from IPython.display import HTML, display
import genomic_benchmarks_data4agents as gbd


def display_qc_report(path, height=900):
    """Display a standalone QC report without leaking its styles into Jupyter."""
    document = escape(path.read_text(encoding="utf-8"), quote=True)
    return HTML(
        f"<iframe srcdoc=\"{document}\" "
        f"style=\"width: 100%; height: {height}px; border: 1px solid #ddd;\""
        "></iframe>"
    )

## Discover datasets

In [ ]:
collections = gbd.list_collections()
binary_datasets = gbd.list_datasets(task_type="Binary classification")

collections, binary_datasets

The examples below use a small dataset with train and `val` data.

In [ ]:
COLLECTION = "BEACON"
DATASET = "CRISPROnTarget"

## Inspect metadata

In [ ]:
dataset_info = gbd.get_dataset_info(COLLECTION, DATASET)
validation_info = gbd.get_split_info(COLLECTION, DATASET, "val")

{
    "task_type": dataset_info["task_type"],
    "sequence_columns": dataset_info["seq_col"],
    "label_column": dataset_info["label_col"],
    "available_splits": dataset_info["splits"],
    "val_size": validation_info["dataset_size"],
}

Only the splits listed above are accessible. Test and `test_leftout` files and their metadata are not included in this package.

## Load train and validation data

By default, `load_dataset()` returns the sequence and label columns.

In [ ]:
train = gbd.load_dataset(COLLECTION, DATASET, "train")
validation = gbd.load_dataset(COLLECTION, DATASET, "val")

{"train_columns": train.columns.tolist(), "val_columns": validation.columns.tolist()}

In [ ]:
validation.head()

Use `columns="all"` when identifiers or other auxiliary columns are needed.

In [ ]:
validation_all = gbd.load_dataset(
    COLLECTION, DATASET, "val", columns="all"
)
validation_all.columns.tolist()

## Locate a downloaded file

`download_dataset()` returns the complete cached file, regardless of the column view selected by `load_dataset()`.

In [ ]:
cached_validation_path = gbd.download_dataset(COLLECTION, DATASET, "val")
cached_validation_path

## Explore QC results

Class QC is available for train and `val`. Split QC is limited to the train–`val` comparison; datasets without `val` have no split-QC result.

In [ ]:
class_qc = gbd.load_qc_results("class")
split_qc = gbd.load_qc_results("split")

{
    "class_checks": gbd.list_qc_metrics("class"),
    "split_checks": gbd.list_qc_metrics("split"),
}

In [ ]:
passing_class_qc = gbd.filter_datasets_by_qc("class", flag="Pass")
passing_split_qc = gbd.filter_datasets_by_qc("split", flag="Pass")

{"class": passing_class_qc, "split": passing_split_qc}

In [ ]:
gbd.plot_qc_summary("split")

## Open self-contained QC reports

In [ ]:
class_report = gbd.get_class_qc_report(
    COLLECTION, DATASET, split="val", seq_col="seq"
)
display(display_qc_report(class_report))

In [ ]:
split_report = gbd.get_split_qc_report(
    COLLECTION, DATASET, split_a="train", split_b="val"
)
display(display_qc_report(split_report))